# Data Preprocessing: 
## Merging and Selecting Features

## Objective
Create a unified dataset of **new** and **used** cars with only the most relevant features for analysis and modeling.

## Steps

1. **Merge Datasets**
   - Combine **new** and **used** car datasets.
   - Add a new column `new`:
     - `"yes"` for new cars  
     - `"no"` for used cars  

2. **Select Relevant Features**
   - Columns retained based on previous **Data Quality Assessment** and feature relevance:
     - **Price** – convert new cars to numeric  
     - **Brand**  
     - **Model**  
     - **Kilométrage** – set to `0` for new cars  
     - **Body type** (`Carrosserie`)  
     - **Fuel type** (`Energie`)  
     - **Seats** (`Nombre de places`)  
     - **Transmission** (`Boîte` / `Boite vitesse`)  
     - **Puissance fiscale** – used instead of `Puissance en chevaux` due to missing data  
     - **Date mise en circulation** – set to `2025` for new cars  
     - **Climatisation** – auto or manual  
     - **New flag** – added in merging step  

3. **Rationale for Eliminations**
   - Columns removed due to:
     - Excessive missing data  
     - Redundancy (e.g., number of doors vs. seats, car dimensions vs. body type)  
     - Irrelevance or low predictive value (e.g., detailed motor specifications, fuel consumption metrics)

## Outcome
A clean, consolidated dataset with key features, ready for preprocessing steps like encoding, missing value handling, and scaling.


In [360]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load data
df_new = pd.read_csv("../data_scraper/data_scraped/automobileTnNew.csv")
df_used = pd.read_csv("../data_scraper/data_scraped/automobileTnUsed.csv")

def clean_year_string(year_val):
    if pd.isna(year_val):
        return None
    # Convert to string and handle float representation
    year_str = str(year_val)
    if '.' in year_str:
        parts = year_str.split('.')
        if len(parts) == 2:
            month = parts[0]
            year_part = parts[1]
            if len(year_part) == 3:
                year_part = year_part + '0' 
            return f"{month}.{year_part}"
    return year_str

df_used['Mise en circulation'] = df_used['Mise en circulation'].apply(clean_year_string)

# Adding the 'new' column to each dataset
df_used['new'] = 'no'
df_new['new'] = 'yes'

# Preparing new cars dataset
df_new['Price_clean'] = df_new['Price'].str.replace(' ', '').astype(float)

# Process climatisation for new cars
def check_auto_climatisation(clim_str):
    if pd.isna(clim_str):
        return 'No'
    clim_str = str(clim_str).lower()
    if 'automatique' in clim_str:
        return 'Yes'
    else:
        return 'no'

df_new['Climatisation_auto'] = df_new['Climatisation'].apply(check_auto_climatisation)

# Add missing columns for new cars with default values
df_new['Kilométrage'] = 0  
df_new['Mise en circulation'] = '12.2025' 

# Select and rename columns for new cars
df_new_clean = df_new[['Brand', 'Model', 'Price_clean', 'Kilométrage', 'Carrosserie', 
                      'Energie', 'Nombre de places', 'Boîte', 'Puissance fiscale', 
                      'Mise en circulation', 'Climatisation_auto', 'new']].copy()

df_new_clean = df_new_clean.rename(columns={
    'Brand': 'brand',
    'Model': 'model', 
    'Price_clean': 'price',
    'Kilométrage': 'kilometrage',
    'Carrosserie': 'body_type',
    'Energie': 'fuel_type',
    'Nombre de places': 'seats',
    'Boîte': 'transmission',
    'Puissance fiscale': 'puissance_fiscale',
    'Mise en circulation': 'registration_year',
    'Climatisation_auto': 'climatisation'
})

# Preparing used cars dataset
# Select and rename columns for used cars
df_used_clean = df_used[['Spécifications - Marque', 'Spécifications - Modèle', 'Price', 
                        'Kilométrage', 'Carrosserie', 'Motorisation - Énergie',
                        'Spécifications - Nombre de places', 'Boite vitesse', 'Puissance fiscale',
                        'Mise en circulation', 'Fonctionnels - Climatisation automatique', 'new']].copy()

df_used_clean = df_used_clean.rename(columns={
    'Spécifications - Marque': 'brand',
    'Spécifications - Modèle': 'model',
    'Price': 'price', 
    'Kilométrage': 'kilometrage',
    'Carrosserie': 'body_type',
    'Motorisation - Énergie': 'fuel_type',
    'Spécifications - Nombre de places': 'seats',
    'Boite vitesse': 'transmission',
    'Puissance fiscale': 'puissance_fiscale',
    'Mise en circulation': 'registration_year',
    'Fonctionnels - Climatisation automatique': 'climatisation'
})

# Merging the datasets
merged_df = pd.concat([df_used_clean, df_new_clean], ignore_index=True)

# Display results
print(f"Final merged dataset shape: {merged_df.shape}")
print(f"Used cars: {(merged_df['new'] == 'no').sum()}")
print(f"New cars: {(merged_df['new'] == 'yes').sum()}")

# VERIFY THE FIX
print(f"\n=== REGISTRATION YEAR DISTRIBUTION (FIXED) ===")
print("Top 20 registration_year values:")
print(merged_df['registration_year'].value_counts().head(20))

print("\nFirst 15 rows of merged dataset:")
print(merged_df.head(15))

Final merged dataset shape: (2658, 12)
Used cars: 2148
New cars: 510

=== REGISTRATION YEAR DISTRIBUTION (FIXED) ===
Top 20 registration_year values:
12.2025    510
5.2022      37
11.2021     36
3.2021      34
1.2021      33
9.2020      30
9.2021      28
3.2022      28
1.2022      27
1.2020      27
6.2021      27
2.2021      26
6.2022      26
7.2020      25
7.2021      25
10.2021     24
11.2019     24
10.2020     24
12.2020     23
8.2021      22
Name: registration_year, dtype: int64

First 15 rows of merged dataset:
            brand         model     price kilometrage body_type  \
0             GWM  Haval Jolion   78000.0  113 000 km       SUV   
1            Audi            A6   75000.0  160 000 km   Berline   
2   Mercedes-Benz     GLE Coupé  460000.0   15 000 km       SUV   
3   Mercedes-Benz      Classe E  118000.0  140 000 km   Berline   
4       Chevrolet        Groove   68000.0   67 000 km       SUV   
5           Skoda         Fabia   47000.0  115 000 km  Citadine   
6        

## Cleaning Rows and Removing Outliers

## Objective
Refine the dataset by removing irrelevant or extreme entries to improve quality and consistency.

## Steps

1. **Filter by Fuel Type**
   - Keep only cars with fuel type:
     - `Essence` (petrol)
     - `Diesel`
     - `Hybride` (hybrid)
     - `Electrique` (electric)
   - combine all variations of hybrid into one hybrid class.

2. **Filter by Brand and Body Type**
   - Remove cars and brands with **very low representation** to avoid sparsity and unreliable patterns.
   - Apply the same principle to **body types** to focus on the most common categories.

3. **Remove Outliers**
   - **Price**: remove extreme low or high values outside expected range.  
   - **Year of circulation** (`Date mise en circulation`): remove unrealistic years.  
   - **Kilométrage**: remove extreme mileage values that are likely data errors.

## Outcome
A cleaner, more representative dataset with consistent fuel types, brands, body types, and realistic numerical values, ready for further preprocessing like encoding and scaling.


In [361]:
# DATA CLEANING And ROW FILTERING

print(f"Initial dataset shape: {merged_df.shape}")

# Create a clean copy
merged_clean = merged_df.copy()

# 1. Fuel Type Transformation : simplify categories and remove Compacte (it has one row only)
fuel_counts = merged_clean['fuel_type'].value_counts()
print("\nFuel type distribution:")
print(fuel_counts)

# Remove the "Compacte" row first
merged_clean = merged_clean[merged_clean['fuel_type'] != 'Compacte'].copy()

# Transform fuel types into simplified categories
def simplify_fuel_type(fuel):
    fuel = str(fuel).lower()
    if 'electrique' in fuel:
        return 'electrique'
    elif 'hybride' in fuel:
        return 'hybride' 
    else:
        return fuel  

merged_clean['fuel_type_simple'] = merged_clean['fuel_type'].apply(simplify_fuel_type)

# Keep all simplified fuel types except rare ones
fuel_types_to_keep = ['essence', 'diesel', 'hybride', 'electrique']
merged_clean = merged_clean[merged_clean['fuel_type_simple'].isin(fuel_types_to_keep)].copy()
print(f"After fuel filtering: {merged_clean.shape}")

print("\nSimplified fuel type distribution:")
print(merged_clean['fuel_type_simple'].value_counts())

# 2. Brand Filtering : remove brands with low representation
brand_counts = merged_clean['brand'].value_counts()
print(f"\nBrand counts:")
print(brand_counts)

# Keep brands with at least 10 cars
brands_to_keep = brand_counts[brand_counts >= 10].index
merged_clean = merged_clean[merged_clean['brand'].isin(brands_to_keep)].copy()
print(f"After brand filtering: {merged_clean.shape}")

# 3. Body Type Filtering : remove rare body types
body_counts = merged_clean['body_type'].value_counts()
print(f"\nBody type counts:")
print(body_counts)

# Keep body types with at least 30 cars
bodies_to_keep = body_counts[body_counts >= 30].index
merged_clean = merged_clean[merged_clean['body_type'].isin(bodies_to_keep)].copy()
print(f"After body type filtering: {merged_clean.shape}")

# 4. Price Outlier Removal
price_stats = merged_clean['price'].describe()
print(f"\nPrice statistics before:")
print(price_stats)

# Remove extreme price outliers (bottom and top 1%)
Q1_price = merged_clean['price'].quantile(0.01)
Q3_price = merged_clean['price'].quantile(0.99)
merged_clean = merged_clean[(merged_clean['price'] >= Q1_price) & (merged_clean['price'] <= Q3_price)].copy()
print(f"After price filtering: {merged_clean.shape}")


Initial dataset shape: (2658, 12)

Fuel type distribution:
Essence                           1707
Diesel                             557
Hybride rechargeable essence       127
Electrique                          95
Hybride léger essence               60
Essence | Hybride rechargeable      34
Essence | Hybride léger             25
Essence | Hybride                   23
Hybride essence                     14
Hybride léger diesel                10
Hybride rechargeable diesel          3
Diesel | Hybride léger               2
Compacte                             1
Name: fuel_type, dtype: int64
After fuel filtering: (2657, 13)

Simplified fuel type distribution:
essence       1707
diesel         557
hybride        298
electrique      95
Name: fuel_type_simple, dtype: int64

Brand counts:
Mercedes-Benz    397
Volkswagen       204
KIA              166
BMW              151
Peugeot          141
                ... 
Lexus              1
BAIC YX            1
Tata               1
Chrysler          

In [ ]:
# 5. Year Filtering : 
print(f"=== YEAR FILTERING ===")

# Extract year from registration_year
def extract_year_fixed(year_str):
    if pd.isna(year_str):
        return None
    year_str = str(year_str).strip()
    if '.' in year_str:
        parts = year_str.split('.')
        if len(parts) == 2:
            year_part = parts[1]
            try:
                year = int(year_part)
                if 1900 <= year <= 2030:
                    return year
            except:
                return None
    return None

merged_clean['year_extracted'] = merged_clean['registration_year'].apply(extract_year_fixed)

# Remove cars outside 2007-2025 range and null years
current_year = 2025
initial_count = len(merged_clean)
merged_clean = merged_clean[
    (merged_clean['year_extracted'] >= 2007) & 
    (merged_clean['year_extracted'] <= current_year) &
    (merged_clean['year_extracted'].notnull())
].copy()

print(f"Rows removed by year filtering: {initial_count - len(merged_clean)}")
print(f"Final dataset shape: {merged_clean.shape}")

=== YEAR FILTERING ===
Rows removed by year filtering: 35
Final dataset shape: (2336, 14)


In [363]:
# 6. Mileage Outlier Removal

# Convert kilometrage properly : removing 'km' and spaces, then converting to float
merged_clean['kilometrage'] = merged_clean['kilometrage'].astype(str).str.replace(' km', '').str.replace(' ', '').astype(float)

# Remove unrealistic mileages  while keeping new cars (0 km)
merged_clean = merged_clean[(merged_clean['kilometrage'] <= 250000) | (merged_clean['new'] == 'yes')].copy()
print(f"After mileage filtering: {merged_clean.shape}")

#  summary
print(f"\n=== CLEANING SUMMARY ===")
print(f"Initial rows: {len(merged_df)}")
print(f"Final rows: {len(merged_clean)}")
print(f"Rows removed: {len(merged_df) - len(merged_clean)}")
print(f"Removal percentage: {((len(merged_df) - len(merged_clean)) / len(merged_df) * 100):.1f}%")


merged_clean.to_csv('car_prices_cleaned.csv', index=False)



After mileage filtering: (2230, 14)

=== CLEANING SUMMARY ===
Initial rows: 2658
Final rows: 2230
Rows removed: 428
Removal percentage: 16.1%


In [364]:
#checking for empty values
print(merged_clean.isnull().sum())

# Checking categorical variables
print("Categorical variables value counts:")
for col in ['brand', 'body_type', 'fuel_type_simple', 'transmission', 'climatisation', 'new']:
    print(f"\n{col}:")
    print(merged_clean[col].value_counts())


brand                0
model                0
price                0
kilometrage          0
body_type            0
fuel_type            0
seats                0
transmission         0
puissance_fiscale    0
registration_year    0
climatisation        0
new                  0
fuel_type_simple     0
year_extracted       0
dtype: int64
Categorical variables value counts:

brand:
Mercedes-Benz    364
Volkswagen       182
KIA              155
BMW              130
Peugeot          130
Audi             126
Hyundai           65
Land Rover        64
Ford              60
Seat              51
Toyota            50
bmw               49
GWM               49
Citroën           46
Renault           45
MG                41
mercedes-benz     34
hyundai           33
Nissan            31
Porsche           31
Jeep              31
Fiat              31
Chery             30
Suzuki            30
Mazda             26
kia               22
peugeot           20
Skoda             19
Dacia             17
mg          

## Standardization and Feature Engineering

##  Brand Name Standardization
- Correct inconsistencies in brand names using a mapping dictionary.
- Example: `'bmw' → 'BMW'`, `'mercedes-benz' → 'Mercedes-Benz'`.

##  Climatisation Standardization
- Harmonize values to lowercase: `'Yes' → 'yes'`, `'No' → 'no'`.

##  Fuel Type Update
- Remove the old `fuel_type` column.
- Rename `fuel_type_simple` to `fuel_type` for clarity.

##  Extract Month from Registration Year
- Added a `month` column based on `registration_year`.
- Handles missing or improperly formatted values.

##  Puissance Fiscale Conversion
- Convert `puissance_fiscale` from string to numeric.
- Extract the first number from the string for accurate calculations.

##  Feature Engineering
- `car_age`: `2025 - year_extracted` to calculate vehicle age.
- `is_new`: binary flag for new cars (`1` = yes, `0` = no).
- `price_per_fiscal`: computed as `price / puissance_fiscale` for normalized pricing.

##  Binary Categoricals Conversion
- Convert categorical columns to 0/1:
  - `climatisation`: `yes = 1`, `no = 0`
  - `new`: `yes = 1`, `no = 0`
  - `transmission`: `Automatique = 1`, `Manuelle = 0`



In [365]:
# fix brand name inconsistencies
brand_mapping = {
    'bmw': 'BMW', 'mercedes-benz': 'Mercedes-Benz', 'hyundai': 'Hyundai',
    'kia': 'KIA', 'peugeot': 'Peugeot', 'toyota': 'Toyota', 
    'volkswagen': 'Volkswagen', 'skoda': 'Skoda', 'suzuki': 'Suzuki',
    'renault': 'Renault', 'audi': 'Audi', 'volvo': 'Volvo',
    'seat': 'Seat', 'honda': 'Honda', 'fiat': 'Fiat', 'citroen': 'Citroën',
    'porsche': 'Porsche', 'mg': 'MG'
}

merged_clean['brand'] = merged_clean['brand'].replace(brand_mapping)

# Fix climatisation inconsistencies  
merged_clean['climatisation'] = merged_clean['climatisation'].replace({'No': 'no', 'Yes': 'yes'})

#  Remove old fuel_type and rename fuel_type_simple
merged_clean = merged_clean.drop('fuel_type', axis=1)
merged_clean = merged_clean.rename(columns={'fuel_type_simple': 'fuel_type'})

#  Add month column from registration_year
def extract_month(year_str):
    if pd.isna(year_str):
        return None
    year_str = str(year_str)
    if '.' in year_str:
        return int(year_str.split('.')[0])
    return None

merged_clean['month'] = merged_clean['registration_year'].apply(extract_month)
def extract_fiscal_power(value):
    if pd.isna(value):
        return None
    value_str = str(value)
    # Extract first number from string (ex : 5 CV -> 5)
    import re
    numbers = re.findall(r'\d+', value_str)
    if numbers:
        return float(numbers[0])
    return None

# 5. Convert puissance_fiscale to numeric 
merged_clean['puissance_fiscale'] = merged_clean['puissance_fiscale'].apply(extract_fiscal_power)

# 6. Feature Engineering
merged_clean['car_age'] = 2025 - merged_clean['year_extracted']
merged_clean['is_new'] = (merged_clean['new'] == 'yes').astype(int)
merged_clean['price_per_fiscal'] = merged_clean['price'] / merged_clean['puissance_fiscale']

print(f"Dataset shape: {merged_clean.shape}")
print(f"New columns: month, car_age, is_new, price_per_fiscal")
print(f"Columns: {merged_clean.sample(10)}")

Dataset shape: (2230, 17)
New columns: month, car_age, is_new, price_per_fiscal
Columns:               brand           model     price  kilometrage   body_type seats  \
333         Citroën       C4 Cactus   58000.0     148000.0    Compacte     5   
1324  Mercedes-Benz        Classe E  155000.0     130000.0     Berline     5   
1004         Nissan         Qashqai   47500.0     175000.0         SUV     5   
1413        Citroën    Berlingo Van   55000.0     155000.0  Utilitaire     3   
1237          Mazda               6   48000.0     157000.0     Berline     5   
1083          Chery     Tiggo 7 Pro   82000.0      35484.0         SUV     5   
779   Mercedes-Benz        Classe C  110000.0      87000.0     Berline     4   
275   Mercedes-Benz  Classe C coupé   72000.0     100000.0       Coupé     4   
826      Volkswagen          Passat   89000.0     190000.0     Berline     5   
1121  Mercedes-Benz        Classe C  258000.0       5686.0     Berline     5   

     transmission  puissance_f

In [366]:
# Convert binary categoricals to 0 or 1
binary_mapping = {
    'climatisation': {'yes': 1, 'no': 0},
    'new': {'yes': 1, 'no': 0},
    'transmission': {'Automatique': 1, 'Manuelle': 0}
}

for col, mapping in binary_mapping.items():
    merged_clean[col] = merged_clean[col].map(mapping)

# checking conversions
print("Binary conversions:")
for col in ['climatisation', 'new', 'transmission']:
    print(f"{col}: {merged_clean[col].value_counts()}")

print(f"\nDataset shape: {merged_clean.shape}")
print(f"Columns: {merged_clean.sample(10)}")

Binary conversions:
climatisation: 1    1226
0    1004
Name: climatisation, dtype: int64
new: 0    1887
1     343
Name: new, dtype: int64
transmission: 1    1439
0     791
Name: transmission, dtype: int64

Dataset shape: (2230, 17)
Columns:               brand              model     price  kilometrage body_type seats  \
2090          Mazda              BT-50   50000.0     162000.0   Pick up     5   
156         Peugeot               3008   63000.0      85000.0       SUV     5   
1370     Volkswagen             Golf 8   78000.0      79000.0  Compacte     5   
866         Hyundai             Tucson  105000.0     113000.0       SUV     5   
305             KIA            Picanto   43000.0      81000.0  Citadine     5   
88          Hyundai           Palisade  250000.0      35774.0       SUV     7   
943      Land Rover  Range Rover Sport  350000.0      82000.0       SUV     5   
1010     Volkswagen             Passat   40000.0     180000.0   Berline     5   
1147  Mercedes-Benz           

## Encoding, Scaling, and Final Cleaning

##  Target Encoding
- Encode categorical variables (`brand`, `body_type`, `model`) using mean `price` per category.
- Created separate CSVs for reference:
  - `brand_encoding.csv`
  - `body_type_encoding.csv`
  - `model_encoding.csv`
- Original columns dropped after encoding to avoid redundancy.

##  Feature Scaling
- Standardized numerical features for modeling using `StandardScaler`:
  - `kilometrage`, `puissance_fiscale`, `car_age`, `price_per_fiscal`, `month`
  - Encoded features: `brand_encoded`, `body_type_encoded`, `model_encoded`
- Ensures features are comparable in scale.

##  One-Hot Encoding
- Applied one-hot encoding to `fuel_type` for categorical modeling.
- Prefix `fuel_` added to new columns.

##  Redundant Column Removal
- Dropped columns replaced by engineered features:
  - `year_extracted` → `car_age`
  - `new` → `is_new`
  - `registration_year` → month already extracted
- Only columns existing in the dataset were dropped.

##  Missing Value Handling
- Identified rows with missing values.
- Only 6 rows out of 2230 had missing data.
- Dropped these rows to maintain clean dataset.

##  Final Dataset
- Preprocessed, scaled, and encoded dataset ready for modeling.
- Saved to `car_prices_final_preprocessed.csv`.


In [367]:
def manual_target_encode(series, target):
    return series.map(target.groupby(series).mean())

brand_encoding_map = merged_clean.groupby('brand')['price'].mean().to_dict()
body_type_encoding_map = merged_clean.groupby('body_type')['price'].mean().to_dict()
model_encoding_map = merged_clean.groupby('model')['price'].mean().to_dict()

brand_encoding_df = pd.DataFrame({
    'brand': list(brand_encoding_map.keys()),
    'brand_encoded': list(brand_encoding_map.values())
})

body_type_encoding_df = pd.DataFrame({
    'body_type': list(body_type_encoding_map.keys()),
    'body_type_encoded': list(body_type_encoding_map.values())
})

model_encoding_df = pd.DataFrame({
    'model': list(model_encoding_map.keys()),
    'model_encoded': list(model_encoding_map.values())
})

brand_encoding_df.to_csv('brand_encoding.csv', index=False)
body_type_encoding_df.to_csv('body_type_encoding.csv', index=False)
model_encoding_df.to_csv('model_encoding.csv', index=False)

merged_clean['brand_encoded'] = manual_target_encode(merged_clean['brand'], merged_clean['price'])
merged_clean['body_type_encoded'] = manual_target_encode(merged_clean['body_type'], merged_clean['price'])
merged_clean['model_encoded'] = manual_target_encode(merged_clean['model'], merged_clean['price'])

merged_clean = merged_clean.drop(['brand', 'body_type', 'model', 'registration_year'], axis=1)

print(f"Dataset shape: {merged_clean.shape}")

Dataset shape: (2230, 16)


In [368]:
from sklearn.preprocessing import StandardScaler

cols_to_scale = [
    'kilometrage',           
    'puissance_fiscale',    
    'car_age',              
    'price_per_fiscal',     
    'month',                
    'brand_encoded',      
    'body_type_encoded',   
    'model_encoded'        
]

scaler = StandardScaler()

# scale the selected columns
merged_clean[cols_to_scale] = scaler.fit_transform(merged_clean[cols_to_scale])

print("Scaled dataset sample:")
print(merged_clean.sample(5).round(3))

Scaled dataset sample:
         price  kilometrage seats  transmission  puissance_fiscale  \
154   310000.0       -0.551     5             1             -0.062   
635    56000.0        1.579     5             1              0.657   
887   200000.0        0.379     5             0             -0.062   
2217   58950.0       -1.346     5             0             -0.541   
1920   55000.0        1.099     5             0              0.178   

      climatisation  new   fuel_type  year_extracted  month  car_age  is_new  \
154               0    0  electrique            2022 -0.871   -0.570       0   
635               0    0      diesel            2009  1.004    2.567       0   
887               0    0      diesel            2014 -1.139    1.361       0   
2217              0    1     essence            2025  1.271   -1.294       1   
1920              1    0      diesel            2018 -0.603    0.395       0   

      price_per_fiscal  brand_encoded  body_type_encoded  model_encoded  
1

In [369]:

# nne-hot encode fuel_type
merged_clean = pd.get_dummies(merged_clean, columns=['fuel_type'], prefix='fuel')

# removing redundant columns
columns_to_drop = [
    'year_extracted',    # Replaced by car_age
    'new',               # Replaced by is_new 
]


merged_clean = merged_clean.drop(columns_to_drop, axis=1)

print(f"Final dataset shape: {merged_clean.shape}")
print(f"Final columns: {merged_clean.columns.tolist()}")

print("\nFinal dataset sample:")
print(merged_clean.sample(5).round(3))

Final dataset shape: (2230, 17)
Final columns: ['price', 'kilometrage', 'seats', 'transmission', 'puissance_fiscale', 'climatisation', 'month', 'car_age', 'is_new', 'price_per_fiscal', 'brand_encoded', 'body_type_encoded', 'model_encoded', 'fuel_diesel', 'fuel_electrique', 'fuel_essence', 'fuel_hybride']

Final dataset sample:
         price  kilometrage seats  transmission  puissance_fiscale  \
56     35000.0        0.851     4             0             -1.021   
930    65000.0        1.204     3             0             -0.302   
1243   52000.0       -0.326     5             1             -0.541   
553   105000.0       -0.956     5             1             -0.302   
2169   34900.0       -1.346     2             0             -0.781   

      climatisation  month  car_age  is_new  price_per_fiscal  brand_encoded  \
56                0 -0.603    1.119       0            -0.611         -0.649   
930               0 -1.675    0.154       0            -0.536         -1.140   
1243      

In [ ]:
# Handle the missing values
missing_mask = merged_clean.isnull().any(axis=1)
print(f"rows with missing values: {missing_mask.sum()}")

# Show which rows have missing values
if missing_mask.sum() > 0:
    print("\nRows with missing values:")
    print(merged_clean[missing_mask])
    
    # Remove rows with missing values 
    merged_clean = merged_clean.dropna()
    print(f"\nRemoved {missing_mask.sum()} rows with missing values")
    print(f"Final clean dataset: {merged_clean.shape[0]} rows, {merged_clean.shape[1]} columns")
else:
    print("No missing values found!")

#extract csv
merged_clean.to_csv('car_prices_final_preprocessed.csv', index=False)
print("Final preprocessed dataset sample:")
print(merged_clean.sample(5).round(3))



Finding missing values...
Rows with missing values: 3

Rows with missing values:
         price  kilometrage seats  transmission  puissance_fiscale  \
2531  249900.0    -1.345653     5             1                NaN   
2553  438000.0    -1.345653     5             1                NaN   
2597  399900.0    -1.345653     5             1                NaN   

      climatisation     month   car_age  is_new  price_per_fiscal  \
2531              1  1.271464 -1.293929       1               NaN   
2553              1  1.271464 -1.293929       1               NaN   
2597              1  1.271464 -1.293929       1               NaN   

      brand_encoded  body_type_encoded  model_encoded  fuel_diesel  \
2531       1.226440           0.764927       2.001368            0   
2553       1.877001           0.764927       3.488239            0   
2597       1.052263           0.764927       3.906708            0   

      fuel_electrique  fuel_essence  fuel_hybride  
2531                1       

# Summary

We merged new and used car datasets and standardized key features, ensuring consistency in brand names, transmission, and climatisation values. Irrelevant columns and rare categories were removed, and outliers in price, mileage, and year were filtered out. New features were created, including car age, price per fiscal power, and binary flags for new cars and climatisation. Categorical variables were encoded using target encoding and one-hot encoding, while numerical features were scaled for modeling. After handling a few missing values, the final clean dataset is ready for analysis and saved as `car_prices_final_preprocessed.csv`.
